In [21]:
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset, CSGridMLM_collate_fn
from torch.utils.data import DataLoader
from models import SEFiLMModel, ContrastiveSpaceModel, contrastive_normalized_loss
import torch
from torch.nn import CrossEntropyLoss
import os
import pickle
from data_utils import ContrastiveCollator
from train_utils import make_mixed_batch, full_to_partial_masking
from tqdm import tqdm

In [2]:
source_name = 'lstm'
shared_dim = 64
batch_size = 16
device_name = 'cuda:0'

In [3]:
source_key = source_name + '_embeddings'

train_path = 'data/contrastive_dataset/CA_train.pickle'
val_path = 'data/contrastive_dataset/CA_test.pickle'

with open(train_path, 'rb') as f:
    train_dataset = pickle.load(f)
with open(val_path, 'rb') as f:
    val_dataset = pickle.load(f)

source_dim = train_dataset[0][source_key].shape[0]
transformer_dim = train_dataset[0]['transformer_embeddings'].shape[0]

print(source_name, ' - source_dim: ', source_dim)
print('transformer', ' - transformer_dim: ', transformer_dim)
print('shared', ' - shared_dim: ', shared_dim)

collator = ContrastiveCollator(pad_id=0)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)

if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

contrastive_model = ContrastiveSpaceModel(source_dim, transformer_dim, shared_dim=shared_dim)
checkpoint = torch.load(f'saved_models/contrastive/{source_name}.pt', map_location=device_name)
contrastive_model.load_state_dict(checkpoint)
contrastive_model.to(device)

# contrastive_loss_fn = contrastive_normalized_loss
contrastive_loss_fn = torch.nn.MSELoss()

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

logits_loss_fn =CrossEntropyLoss(ignore_index=-100)

transformer_model = SEFiLMModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=512,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=shared_dim,
    device=device,
)
checkpoint = torch.load(f'saved_models/lacta/{source_name}.pt', map_location=device_name)
transformer_model.load_state_dict(checkpoint)
transformer_model.to(device)

lstm  - source_dim:  128
transformer  - transformer_dim:  512
shared  - shared_dim:  64


SEFiLMModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithFiLM(
      (layer): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
      (film): FiLMAdapter(
        (gamma): Linear(in_features=64, out_features=512, bias=True)
        (beta): Line

In [15]:
batch = next(iter(val_loader))

In [16]:
melody_grid = batch["pianoroll"].to(device)
harmony_gt = batch["harmony_ids"].to(device)
home_guidance_embeddings = batch[source_key].to(device)
mixed_batch = make_mixed_batch(batch, source_key)
foreign_guidance_embeddings = mixed_batch[source_key].to(device)

In [17]:
num_visible = -1

harmony_input, harmony_target = full_to_partial_masking(
    harmony_gt,
    mask_token_id,
    num_visible,
    bar_token_id=bar_token_id
)

# Step 1: contrastive latent attraction validation
z_guidance = contrastive_model.source_proj(foreign_guidance_embeddings.to(device))
logits, hidden = transformer_model(
    melody_grid.to(device),
    harmony_input.to(device),
    z_guidance.to(device),
    return_hidden=True
)
z_guidance, z_transformer, _ = contrastive_model( foreign_guidance_embeddings.to(device), hidden.to(device))
foreign_guidance_loss = contrastive_loss_fn(z_guidance,z_transformer)

# Step 2: home attraction validation
z_guidance = contrastive_model.source_proj(home_guidance_embeddings.to(device))
logits, hidden = transformer_model(
    melody_grid.to(device),
    harmony_input.to(device),
    z_guidance.to(device),
    return_hidden=True
)
z_guidance, z_transformer, _ = contrastive_model( home_guidance_embeddings.to(device), hidden.to(device))
home_guidance_loss = contrastive_loss_fn(z_guidance,z_transformer)

home_logits_loss = logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1))

# Step 3: no attraction
logits, hidden = transformer_model(
    melody_grid.to(device),
    harmony_input.to(device),
    None,
    return_hidden=True
)
z_guidance, z_transformer, _ = contrastive_model( home_guidance_embeddings.to(device), hidden.to(device))
no_guidance_to_home_loss = contrastive_loss_fn(z_guidance,z_transformer)

no_guidance_to_home_logits_loss = logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1))

z_guidance, z_transformer, _ = contrastive_model( foreign_guidance_embeddings.to(device), hidden.to(device))
no_guidance_to_foreign_loss = contrastive_loss_fn(z_guidance,z_transformer)

In [19]:
print('foreign_guidance_loss: ', foreign_guidance_loss.item())
print('home_guidance_loss: ', home_guidance_loss.item())
print('no_guidance_to_home_loss: ', no_guidance_to_home_loss.item())
print('no_guidance_to_foreign_loss: ', no_guidance_to_foreign_loss.item())
print(10*'-', 'logits', 10*'-')
print('home_logits_loss: ', home_logits_loss.item())
print('no_guidance_to_home_logits_loss: ', no_guidance_to_home_logits_loss.item())

foreign_guidance_loss:  0.0039731645956635475
home_guidance_loss:  0.003247717395424843
no_guidance_to_home_loss:  0.029178807511925697
no_guidance_to_foreign_loss:  0.031460776925086975
---------- logits ----------
home_logits_loss:  0.0194842629134655
no_guidance_to_home_logits_loss:  0.0005857308278791606


In [ ]:
tqdm_position = 0
epoch = 0
step = 0

with torch.no_grad():
    running_loss = 0
    val_loss = 0
    running_accuracy = 0
    val_accuracy = 0

    val_foreign_loss = 0
    val_home_loss = 0
    val_no2home_loss = 0
    val_no2foreign_loss = 0
    running_foreign_loss = 0
    running_home_loss = 0
    running_no2home_loss = 0
    running_no2foreign_loss = 0
    batch_num = 0
    print('validation')
    with tqdm(val_loader, unit='batch', position=tqdm_position) as tepoch:
        tepoch.set_description(f'Epoch {epoch}@{step}| val')
        for batch in tepoch:
            melody_grid = batch["pianoroll"].to(device)
            harmony_gt = batch["harmony_ids"].to(device)
            home_guidance_embeddings = batch[source_key].to(device)
            mixed_batch = make_mixed_batch(batch, source_key)
            foreign_guidance_embeddings = mixed_batch[source_key].to(device)

            harmony_input, harmony_target = full_to_partial_masking(
                harmony_gt,
                mask_token_id,
                num_visible,
                bar_token_id=bar_token_id
            )

            # Step 1: contrastive latent attraction validation
            z_guidance = contrastive_model.source_proj(foreign_guidance_embeddings.to(device))
            logits, hidden = transformer_model(
                melody_grid.to(device),
                harmony_input.to(device),
                z_guidance.to(device),
                return_hidden=True
            )
            z_guidance, z_transformer, _ = contrastive_model( foreign_guidance_embeddings.to(device), hidden.to(device))
            foreign_guidance_loss = contrastive_loss_fn(z_guidance,z_transformer)

            # Step 2: home attraction validation
            z_guidance = contrastive_model.source_proj(home_guidance_embeddings.to(device))
            logits, hidden = transformer_model(
                melody_grid.to(device),
                harmony_input.to(device),
                z_guidance.to(device),
                return_hidden=True
            )
            z_guidance, z_transformer, _ = contrastive_model( home_guidance_embeddings.to(device), hidden.to(device))
            home_guidance_loss = contrastive_loss_fn(z_guidance,z_transformer)

            logits_loss = logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1))

            loss = foreign_guidance_loss + home_guidance_loss + logits_loss

            # update loss and accuracy
            batch_num += 1
            running_loss += loss.item()
            val_loss = running_loss/batch_num
            # accuracy
            predictions = logits.argmax(dim=-1)
            # mask = torch.logical_and(harmony_target != harmony_input, harmony_target != -100)
            mask = harmony_target != -100
            running_accuracy += (predictions[mask] == harmony_target[mask]).sum().item()/mask.sum().item()
            val_accuracy = running_accuracy/batch_num
            
            # partial losses
            running_foreign_loss += foreign_guidance_loss.item()
            val_foreign_loss = running_foreign_loss/batch_num
            running_home_loss += home_guidance_loss.item()
            val_home_loss = running_home_loss/batch_num

            # Step 3: no attraction
            logits, hidden = transformer_model(
                melody_grid.to(device),
                harmony_input.to(device),
                None,
                return_hidden=True
            )
            z_guidance, z_transformer, _ = contrastive_model( home_guidance_embeddings.to(device), hidden.to(device))
            no_guidance_to_home_loss = contrastive_loss_fn(z_guidance,z_transformer)

            no_guidance_to_home_logits_loss = logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1))

            z_guidance, z_transformer, _ = contrastive_model( foreign_guidance_embeddings.to(device), hidden.to(device))
            no_guidance_to_foreign_loss = contrastive_loss_fn(z_guidance,z_transformer)

            tepoch.set_postfix(
                loss=val_loss,
                floss=val_foreign_loss,
                hloss=val_home_loss,
                acc=val_accuracy
            )
        # end for batch
    # end with tqdm
# end with no grad

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 76.41batch/s, acc=0.991, floss=0.00397, hloss=0.0037, loss=0.0393] 
